# Kaggle Titanic Dataset (Logistics regression)

In [ ]:
# 1) Imports and paths
import os
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

DATA_DIR = "/home/atul-kumar/workspace/kaggle/titanic/data"
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")
SUBMISSION_PATH = os.path.join(DATA_DIR, "submission-logistic.csv")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Paths set:", TRAIN_PATH, TEST_PATH)

Paths set: /home/atul-kumar/workspace/kaggle/titanic/data/train.csv /home/atul-kumar/workspace/kaggle/titanic/data/test.csv


In [4]:
# 2) Load data
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
print("Train shape:", train_df.shape, " Test shape:", test_df.shape)
train_df.head(3)

Train shape: (891, 12)  Test shape: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [5]:
# 3) Define features & model pipeline

def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    # Title from Name
    out["Title"] = out["Name"].str.extract(r",\s*([^\.]+)\.")
    # Family size
    out["FamilySize"] = out.get("SibSp", 0) + out.get("Parch", 0) + 1
    # IsAlone flag
    out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
    # Ticket group size as a cheap proxy (count per ticket)
    if "Ticket" in out.columns:
        counts = out["Ticket"].value_counts()
        out["TicketGroup"] = out["Ticket"].map(counts)
    else:
        out["TicketGroup"] = 1
    return out

# Columns to use
base_features = [
    "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked",
]
engineered = ["Title", "FamilySize", "IsAlone", "TicketGroup"]
all_features = base_features + engineered

# Build preprocessing
numeric_features = ["Age", "SibSp", "Parch", "Fare", "FamilySize", "TicketGroup"]
categorical_features = ["Pclass", "Sex", "Embarked", "Title"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# Model
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)

# Full pipeline
model = Pipeline(steps=[
    ("pre", preprocess),
    ("clf", log_reg)
])

# Quick CV check (optional)
train_df_fe = add_engineered_features(train_df)
X = train_df_fe[all_features]
y = train_df_fe["Survived"]
cv_scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
print("CV accuracy:", cv_scores.mean().round(4), "+/-", cv_scores.std().round(4))

CV accuracy: 0.8193 +/- 0.0246


In [ ]:
# 4) Train on full data, predict test, and save submission

# Fit on full training data
model.fit(X, y)

# Prepare test features using the same engineering
test_df_fe = add_engineered_features(test_df)
X_test = test_df_fe[all_features]

# Predict class labels (0/1)
test_pred = model.predict(X_test)

# Ensure test_pred is a NumPy array
if isinstance(test_pred, tuple):
    test_pred = test_pred[0]

# Build submission
submission = pd.DataFrame({
    "PassengerId": test_df_fe["PassengerId"],
    "Survived": test_pred.astype(int)
})

# Save
submission.to_csv(SUBMISSION_PATH, index=False)
print("Saved submission to:", SUBMISSION_PATH)
submission.head(10)

Saved submission to: /home/atul-kumar/workspace/kaggle/titanic/data/submission.csv


,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0
